In [1]:
# Importing the Libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
!pip install torch

In [4]:
!pip install jovian

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for uuid: filename=uuid-1.30-py3-none-any.whl size=6485 sha256=9b7e1ad3c8057c6dd7944201bdfa02e53450e8319f5fd305fa0822dbbc493ad7
  Stored in directory: c:\users\irt\appdata\local\pip\cache\wheels\cc\9d\72\13ff6a181eacfdbd6651ed761a4ee7c5c9f92034a9dc8a1b3c
Successfully built uuid

   ---------------------------------------- 2/2 [jovian]



  DEPRECATION: Building 'uuid' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'uuid'. Discussion can be found at https://github.com/pypa/pip/issues/6334


In [6]:
import torch
import jovian
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, random_split

In [7]:
# Loading the Data
data = pd.read_csv('car data.csv')

In [8]:
# Analyse the Top 5 rows of the Data
data.head()

,Car_Name,Year,Selling_Price,Present_Price,Kms_Driven,Fuel_Type,Seller_Type,Transmission,Owner
0,ritz,2014,3.35,5.59,27000,Petrol,Dealer,Manual,0
1,sx4,2013,4.75,9.54,43000,Diesel,Dealer,Manual,0
2,ciaz,2017,7.25,9.85,6900,Petrol,Dealer,Manual,0
3,wagon r,2011,2.85,4.15,5200,Petrol,Dealer,Manual,0
4,swift,2014,4.60,6.87,42450,Diesel,Dealer,Manual,0


In [9]:
name = "Kamran Ansari"
def customize_dataset(data, rand_str):
    data = data.copy(deep = True)
    data =  data.sample(int(0.95*len(data)), random_state = int(ord(rand_str[0])))
    data.Year = data.Year * ord(rand_str[1]) / 100
    data.Selling_Price = data.Selling_Price * ord(rand_str[2]) / 100
    if ord(rand_str[3]) % 2 == 1:
        data = data.drop(['Car_Name'], axis = 1)
    return data
    
    

In [10]:
data = customize_dataset(data, name)
data.head()

,Car_Name,Year,Selling_Price,Present_Price,Kms_Driven,Fuel_Type,Seller_Type,Transmission,Owner
132,Bajaj Avenger 220,1956.49,0.8175,0.95,3500,Petrol,Individual,Manual,0
280,brio,1954.55,5.7225,5.90,14465,Petrol,Dealer,Manual,0
286,jazz,1955.52,6.1585,7.90,28569,Petrol,Dealer,Manual,0
61,etios cross,1954.55,4.9050,7.70,40588,Petrol,Dealer,Manual,0
242,xcent,1953.58,4.7960,7.13,34000,Petrol,Dealer,Manual,0


In [11]:
input_cols = ["Year","Present_Price","Kms_Driven","Owner"]
categorical_cols = ["Fuel_Type","Seller_Type","Transmission"]
output_cols = ["Selling_Price"]

#### Data Preparation


In [12]:

def dataframe_to_arrays(data):
    # Make a copy of the original dataframe
    dataframe1 = data.copy(deep=True)
    # Convert non-numeric categorical columns to numbers
    for col in categorical_cols:
        dataframe1[col] = dataframe1[col].astype('category').cat.codes
    # Extract input & outupts as numpy arrays
    inputs_array = dataframe1[input_cols].to_numpy()
    targets_array = dataframe1[output_cols].to_numpy()
    return inputs_array, targets_array

inputs_array, targets_array = dataframe_to_arrays(data)
inputs_array, targets_array

(array([[1.95649e+03, 9.50000e-01, 3.50000e+03, 0.00000e+00],
        [1.95455e+03, 5.90000e+00, 1.44650e+04, 0.00000e+00],
        [1.95552e+03, 7.90000e+00, 2.85690e+04, 0.00000e+00],
        ...,
        [1.95358e+03, 8.93000e+00, 8.30000e+04, 0.00000e+00],
        [1.95552e+03, 1.50000e+00, 1.80000e+04, 0.00000e+00],
        [1.95358e+03, 5.59000e+00, 2.70000e+04, 0.00000e+00]]),
 array([[ 0.8175],
        [ 5.7225],
        [ 6.1585],
        [ 4.905 ],
        [ 4.796 ],
        [ 1.0355],
        [ 1.1445],
        [ 0.5668],
        [ 0.2725],
        [ 6.4855],
        [ 0.7085],
        [ 7.3575],
        [ 1.4715],
        [ 6.3765],
        [ 0.654 ],
        [ 5.995 ],
        [ 4.3055],
        [ 4.905 ],
        [ 2.1255],
        [ 5.232 ],
        [ 5.6135],
        [ 0.4578],
        [ 3.379 ],
        [ 0.5232],
        [ 2.8885],
        [ 8.175 ],
        [ 5.0685],
        [ 5.995 ],
        [ 0.981 ],
        [21.5275],
        [ 0.109 ],
        [ 4.905 ],
     

In [13]:

inputs = torch.Tensor(inputs_array)
targets = torch.Tensor(targets_array)

dataset = TensorDataset(inputs, targets)
train_ds, val_ds = random_split(dataset, [228, 57])
batch_size = 128

train_loader = DataLoader(train_ds, batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size)

#### Creating PyTorch Model


In [14]:
input_size = len(input_cols)
output_size = len(output_cols)

class CarsModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(input_size, output_size)                  # fill this (hint: use input_size & output_size defined above)
        
    def forward(self, xb):
        out = self.linear(xb)                          # fill this
        return out
    
    def training_step(self, batch):
        inputs, targets = batch 
        # Generate predictions
        out = self(inputs)          
        # Calcuate loss
        loss = F.l1_loss(out, targets)                         # fill this
        return loss
    
    def validation_step(self, batch):
        inputs, targets = batch
        # Generate predictions
        out = self(inputs)
        # Calculate loss
        loss = F.l1_loss(out, targets)                           # fill this    
        return {'val_loss': loss.detach()}
        
    def validation_epoch_end(self, outputs):
        batch_losses = [x['val_loss'] for x in outputs]
        epoch_loss = torch.stack(batch_losses).mean()   # Combine losses
        return {'val_loss': epoch_loss.item()}
    
    def epoch_end(self, epoch, result, num_epochs):
        # Print result every 20th epoch
        if (epoch+1) % 20 == 0 or epoch == num_epochs-1:
            print("Epoch [{}], val_loss: {:.4f}".format(epoch+1, result['val_loss']))
            
model = CarsModel()

list(model.parameters())

[Parameter containing:
 tensor([[-0.4061, -0.1524,  0.3326,  0.0024]], requires_grad=True),
 Parameter containing:
 tensor([-0.4333], requires_grad=True)]

#### Training Model to Predict Car Prices


In [15]:
# Eval algorithm
def evaluate(model, val_loader):
    outputs = [model.validation_step(batch) for batch in val_loader]
    return model.validation_epoch_end(outputs)

# Fitting algorithm
def fit(epochs, lr, model, train_loader, val_loader, opt_func=torch.optim.SGD):
    history = []
    optimizer = opt_func(model.parameters(), lr)
    for epoch in range(epochs):
        # Training Phase 
        for batch in train_loader:
            loss = model.training_step(batch)
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
        # Validation phase
        result = evaluate(model, val_loader)
        model.epoch_end(epoch, result, epochs)
        history.append(result)
    return history

# Check the initial value that val_loss have
result = evaluate(model, val_loader)
print(result)

{'val_loss': 14027.1767578125}


In [16]:

# Start with the Fitting
epochs = 90
lr = 1e-8
history1 = fit(epochs, lr, model, train_loader, val_loader)

Epoch [20], val_loss: 13413.7344
Epoch [40], val_loss: 12801.2695
Epoch [60], val_loss: 12187.7227
Epoch [80], val_loss: 11573.9033
Epoch [90], val_loss: 11268.3770


In [17]:
# Train repeatdly until have a 'good' val_loss
epochs = 20
lr = 1e-9
history1 = fit(epochs, lr, model, train_loader, val_loader)

Epoch [20], val_loss: 11207.2168


#### Using the Model to Predict Car Prices


In [18]:
# Prediction Algorithm
def predict_single(input, target, model):
    inputs = input.unsqueeze(0)
    predictions = model(inputs)                # fill this
    prediction = predictions[0].detach()
    print("Input:", input)
    print("Target:", target)
    print("Prediction:", prediction)

# Testing the model with some samples
input, target = val_ds[0]
predict_single(input, target, model)

Input: tensor([1.9546e+03, 1.4790e+01, 4.3535e+04, 0.0000e+00])
Target: tensor([12.8075])
Prediction: tensor([10902.4258])


In [19]:

input, target = val_ds[10]
predict_single(input, target, model)

Input: tensor([1.9516e+03, 9.4000e+00, 3.6000e+04, 0.0000e+00])
Target: tensor([4.9050])
Prediction: tensor([8878.4746])
